# FGLS Multi-Model Benchmark
Benchmark geometric compression across multiple GGUF models on Colab T4.

Press **Runtime → Run all** to start.


In [ ]:
#@title 1. Setup Environment { display-mode: "form" }
!bash setup_colab.sh

In [ ]:
#@title 2. Upload & Extract Bundle { display-mode: "form" }
import os
BENCH_DIR = "/content/colab_bench"
if not os.path.exists(BENCH_DIR):
    from google.colab import files
    print("Upload colab_bench.zip:")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    !unzip -qo {fname} -d /content/
else:
    print(f"✓ Bundle already at {BENCH_DIR}")
os.chdir(BENCH_DIR)
import sys; sys.path.insert(0, BENCH_DIR)
print(f"✓ Working directory: {os.getcwd()}")

In [ ]:
#@title 3. Quick Sanity Check { display-mode: "form" }
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
from bench_runner import bench_fibo_addr
ops = bench_fibo_addr(10000)
print(f"fibo_addr: {ops:,.0f} ops/s ✓")

In [ ]:
#@title 4. Run Benchmark { display-mode: "form" }
#@markdown ### Model Selection
models = "qwen3-0.6b-q8" #@param {type:"string"}
#@markdown Comma-separated model IDs, or  for every model
#@markdown ### Benchmark Config
max_tokens = 576 #@param {type:"integer"}
gate_epochs = 500 #@param {type:"integer"}
shell_levels_str = "0,1,2,3" #@param {type:"string"}
shell_levels = [int(x.strip()) for x in shell_levels_str.split(",")]
model_list = None if models.strip().lower() == "all" else [m.strip() for m in models.split(",")]
from bench_runner import bench_all_models, print_comparison_table, save_results_json, plot_comparison
from pathlib import Path
results = bench_all_models(model_ids=model_list, max_tokens=max_tokens, shell_levels=shell_levels, gate_epochs=gate_epochs)
print_comparison_table(results)
save_results_json(results, Path("/content/bench_results.json"))
plot_comparison(results, Path("/content/bench_plots"))

In [ ]:
#@title 5. (Optional) Log to W&B { display-mode: "form" }
use_wandb = False #@param {type:"boolean"}
wandb_project = "fgls-bench" #@param {type:"string"}
if use_wandb:
    from bench_runner import log_to_wandb
    log_to_wandb(results, project=wandb_project)
else:
    print("W&B logging skipped.")

In [ ]:
#@title 6. Download Results { display-mode: "form" }
from google.colab import files
import os, zipfile
with zipfile.ZipFile("/content/bench_results.zip", "w") as zf:
    for f in ["/content/bench_results.json"]:
        if os.path.exists(f): zf.write(f, os.path.basename(f))
    p = "/content/bench_plots"
    if os.path.isdir(p):
        for f in os.listdir(p): zf.write(os.path.join(p, f), f"plots/{f}")
files.download("/content/bench_results.zip")
print("✓ Results downloaded")